# Importing Necessary Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install -q transformers datasets accelerate sentencepiece evaluate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires

In [3]:
from transformers import logging

logging.set_verbosity_warning()   # only important warnings

In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, GroupKFold
from lightgbm import LGBMClassifier, LGBMRanker

from scipy.sparse import hstack
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

import evaluate

from tqdm.auto import tqdm

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

import faiss

# Data Importing

In [5]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

# Model 1 

In [ ]:
train_data = train_df.copy()
test_data = test_df.copy()
submission_data = submission_df.copy()

In [ ]:
print("Dataset preview : \n")
train_data.head()

In [ ]:
print(f"Train data size : {train_data.shape}")
print(f"Test data size : {test_data.shape}")

In [31]:
train_data.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

## Preprocessing

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.replace("\n"," ")
    text = " ".join(text.split())
    return text

In [ ]:
for col in ["prompt","A","B","C","D","E"]:
    train_data[col] = train_data[col].fillna("").apply(clean_text)
    test_data[col] = test_data[col].fillna("").apply(clean_text)

## Convert MCQ into Ranking Format

In [ ]:
options = ["A","B","C","D","E"]

train_rows = []

for _, row in tqdm(train_data.iterrows(), total=len(train_data)):
    
    correct = row["answer"]
    
    for op in options:
        
        train_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[op],
            "option_label": op,
            "label": int(op == correct)
        })

train_long = pd.DataFrame(train_rows)

print(train_long.shape)

## Test Ranking Format

In [ ]:
test_rows = []

for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    
    for op in options:
        
        test_rows.append({
            "id": row["id"],
            "option_label": op,
            "prompt": row["prompt"],
            "option": row[op]
        })

test_long = pd.DataFrame(test_rows)

## Create Additional Features

### Length Features

In [ ]:
for df in [train_long, test_long]:
    df["prompt_len"] = df["prompt"].str.len()
    df["option_len"] = df["option"].str.len()
    df["prompt_words"] = df["prompt"].str.split().str.len()
    df["option_words"] = df["option"].str.split().str.len()

In [ ]:
num_cols = [
    "prompt_len",
    "option_len",
    "prompt_words",
    "option_words"
]

### Word Overlap and Similarity Feature

In [ ]:
def overlap_count(q, a):
    
    q_words = set(q.split())
    a_words = set(a.split())
    
    return len(q_words & a_words)
def overlap_ratio(q, a):

    q_words = set(q.split())
    a_words = set(a.split())

    if len(q_words) == 0:
        return 0

    return len(q_words & a_words) / len(q_words)

In [ ]:
for df in [train_long, test_long]:

    df["overlap"] = [
        overlap_count(q, a)
        for q, a in zip(df["prompt"], df["option"])
    ]
    df["overlap_ratio"] = [
    overlap_ratio(q, a)
    for q, a in zip(df["prompt"], df["option"])
    ]

## TF-IDF Feature Engineering

In [ ]:
train_text = (
    train_long["prompt"] +
    " [SEP] " +
    train_long["option"]
)

test_text = (
    test_long["prompt"] +
    " [SEP] " +
    test_long["option"]
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words="english"
)

In [ ]:
char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    min_df=2,
    max_features=30000
)

In [ ]:
X_char_train = char_vec.fit_transform(train_text)
X_char_test = char_vec.transform(test_text)

In [ ]:
X_word_train = tfidf.fit_transform(train_text)

X_word_test = tfidf.transform(test_text)

### Add Cosine Similarity

In [ ]:
train_prompt_vec = tfidf.transform(train_long["prompt"])
train_option_vec = tfidf.transform(train_long["option"])

test_prompt_vec = tfidf.transform(test_long["prompt"])
test_option_vec = tfidf.transform(test_long["option"])

In [ ]:
train_long["cosine_sim"] = [
    cosine_similarity(
        train_prompt_vec[i],
        train_option_vec[i]
    )[0][0]
    for i in range(len(train_long))
]

test_long["cosine_sim"] = [
    cosine_similarity(
        test_prompt_vec[i],
        test_option_vec[i]
    )[0][0]
    for i in range(len(test_long))
]

In [ ]:
num_cols.extend([
    "overlap",
    "overlap_ratio",
    "cosine_sim"
])

In [ ]:
num_cols

In [ ]:
question_ids = train_long["id"]

In [ ]:
X_num_train = csr_matrix(
    train_long[num_cols].values.astype(np.float32)
)

X_num_test = csr_matrix(
    test_long[num_cols].values.astype(np.float32)
)

In [ ]:
X_train = hstack([
    X_word_train,
    X_char_train,
    X_num_train
]).tocsr()

X_test = hstack([
    X_word_test,
    X_char_test,
    X_num_test
]).tocsr()

In [ ]:
print(type(X_train))
print(type(X_test))

In [ ]:
print(X_word_train.shape)
print(X_char_train.shape)
print(X_num_train.shape)
print(X_train.shape)

In [ ]:
train_long[num_cols].isnull().sum()

In [ ]:
y = train_long["label"]

## LightGBM

### LGBMClassifier

In [ ]:
lgb_params = {
    "n_estimators":1500,
    "learning_rate":0.03,
    "num_leaves":127,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "objective":"binary",
    "metric":"auc",
    "verbosity":-1,
    "random_state":42
}

### Train

In [ ]:
gkf = GroupKFold(n_splits=5)

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

# for fold, (tr_idx, val_idx) in enumerate(
#         skf.split(X_train, y)):
groups = train_long["id"]

for fold, (tr_idx, val_idx) in enumerate(
    gkf.split(X_train, y, groups)
):
    
    print(f"Fold {fold+1}")
    
    model = LGBMClassifier(**lgb_params)
    
    model.fit(
        X_train[tr_idx],
        y.iloc[tr_idx]
    )
    
    oof[val_idx] = model.predict_proba(
        X_train[val_idx]
    )[:,1]
    
    test_preds += (
        model.predict_proba(X_test)[:,1] / 5
    )
    print(test_preds)

In [ ]:
import wandb

run = wandb.init(
    entity="21f3002441-dl-genai-project",
    project="21f3002441-t22026",
    name="lgbm_classifier_tfidf_cv",

    config=lgb_params
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

acc_scores = []
f1_scores = []
auc_scores = []

### LGBMRanker

In [ ]:
ranker_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "force_col_wise": True,
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1
}

In [ ]:
print(X_train.shape)
print(y.value_counts())

### Training

In [ ]:
gkf = GroupKFold(n_splits=5)

groups = train_long["id"]

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(X_train, y, groups)
):

    print(f"Fold {fold+1}")

    ranker = LGBMRanker(**ranker_params)

    # group sizes for training fold
    train_groups = (
        train_long.iloc[tr_idx]
        .groupby("id")
        .size()
        .values
    )

    valid_groups = (
        train_long.iloc[val_idx]
        .groupby("id")
        .size()
        .values
    )

    ranker.fit(
        X_train[tr_idx],
        y.iloc[tr_idx],
        group=train_groups,
        eval_set=[(X_train[val_idx], y.iloc[val_idx])],
        eval_group=[valid_groups]
    )

    oof[val_idx] = ranker.predict(
        X_train[val_idx]
    )

    test_preds += (
        ranker.predict(X_test) / 5
    )

## Save the artifacts

In [ ]:
import joblib

joblib.dump(
    ranker,
    "lgbm_ranker_v1.pkl"
)

joblib.dump(
    tfidf,
    "word_tfidf_v1.pkl"
)

joblib.dump(
    char_vec,
    "char_tfidf_v1.pkl"
)

## Evaluation

In [ ]:
def map3_score(df):

    scores = []

    for _, g in df.groupby("id"):

        pred = (
            g.sort_values(
                "oof_pred",
                ascending=False
            )["option_label"]
            .tolist()[:3]
        )

        true = g.loc[
            g["label"] == 1,
            "option_label"
        ].iloc[0]

        if true == pred[0]:
            scores.append(1.0)

        elif true == pred[1]:
            scores.append(0.5)

        elif true == pred[2]:
            scores.append(1/3)

        else:
            scores.append(0)

    return np.mean(scores)

In [ ]:
train_long["oof_pred"] = oof

cv_score = map3_score(train_long)

print("CV MAP@3 =", cv_score)

In [ ]:
# for _, g in train_long.groupby("id"):
#     print(g[["option_label","label","oof_pred"]])
#     break

In [ ]:
correct = 0

for _, g in train_long.groupby("id"):
    pred = g.loc[g["oof_pred"].idxmax(), "option_label"]
    true = g.loc[g["label"] == 1, "option_label"].iloc[0]

    if pred == true:
        correct += 1

print(correct / train_data.shape[0])

In [ ]:
train_ids = set(train_long.iloc[tr_idx]["id"])
val_ids = set(train_long.iloc[val_idx]["id"])

print(len(train_ids & val_ids))

In [ ]:
for _, g in train_long.groupby("id"):

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values("oof_pred", ascending=False)
    )

    break

In [ ]:
sample_ids = np.random.choice(
    train_long["id"].unique(),
    10,
    replace=False
)

for sid in sample_ids:

    g = train_long[train_long["id"] == sid]

    print("\n", sid)

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values(
            "oof_pred",
            ascending=False
        )
    )

## Predictions

In [ ]:
test_long["score"] = test_preds

In [ ]:
test_long.head()

## Submission Data Preparation for Model 1

In [ ]:
submission_rows = []

for qid, group in test_long.groupby("id"):
    
    ranked = group.sort_values(
        "score",
        ascending=False
    )
    
    top3 = ranked["option_label"].head(3).tolist()
    
    submission_rows.append({
        "id": qid,
        "prediction": " ".join(top3)
    })

In [ ]:
submission = pd.DataFrame(submission_rows)
submission.head()

In [ ]:
submission.to_csv(
    "submission.csv",
    index=False
)

# Model 2 

In [ ]:
train_data = train_df.copy()
test_data = test_df.copy()

print(train_data.shape)
print(test_data.shape)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

## Encode Labels

In [ ]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train_data["label"] = train_data["answer"].map(label_map)

## Train-Validation Split

In [ ]:
train_part, valid_part = train_test_split(
    train_data,
    test_size=0.2,
    stratify=train_data["label"],
    random_state=42
)

## Convert to Hugging Face Dataset

In [ ]:
train_ds = Dataset.from_pandas(train_part)
valid_ds = Dataset.from_pandas(valid_part)

test_ds = Dataset.from_pandas(
    test_data.assign(label=0)
)

## Load DeBERTa Tokenizer

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
MAX_LEN = 256

def preprocess(example):

    first_sentences = [
        example["prompt"]
    ] * 5

    second_sentences = [
        example["A"],
        example["B"],
        example["C"],
        example["D"],
        example["E"]
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN
    )

    tokenized["label"] = example["label"]

    return tokenized

In [ ]:
train_ds = train_ds.map(preprocess)
valid_ds = valid_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

In [ ]:
keep_cols = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "label"
]

train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c not in keep_cols]
)

valid_ds = valid_ds.remove_columns(
    [c for c in valid_ds.column_names if c not in keep_cols]
)

test_ds = test_ds.remove_columns(
    [c for c in test_ds.column_names if c not in keep_cols]
)

In [ ]:
sample = train_ds[0]

print(type(sample["input_ids"]))
print(len(sample["input_ids"]))
print(train_ds.column_names)

for k,v in sample.items():
    print(k, type(v))

## Data Collector

In [ ]:
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

@dataclass
class DataCollatorForMultipleChoice:

    tokenizer: PreTrainedTokenizerBase

    def __call__(self, features):

        labels = [f.pop("label") for f in features]

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = [
            [
                {
                    k: v[i]
                    for k, v in feature.items()
                    if k!= 'label'
                }
                for i in range(num_choices)
            ]
            for feature in features
        ]

        flattened = sum(flattened, [])

        batch = self.tokenizer.pad(
            flattened,
            return_tensors="pt"
        )

        batch = {
            k: v.view(
                batch_size,
                num_choices,
                -1
            )
            for k, v in batch.items()
        }
        if "token_type_ids" in batch:
            del batch["token_type_ids"]
        
        batch["labels"] = torch.tensor(labels)

        return batch

## Load Model

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    dtype = torch.float32
)

In [ ]:
print(next(model.parameters()).dtype)

## Metric

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=1
    )

    return accuracy.compute(
        predictions=preds,
        references=labels
    )

## Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="deberta_mcq",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    num_train_epochs=4,

    fp16=False,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_steps=25,
    
    weight_decay = 0.01,

    report_to="none"
)

In [ ]:
collator = DataCollatorForMultipleChoice(
    tokenizer
)

batch = collator(
    [train_ds[0], train_ds[1]]
)
batch = {k:v.to(model.device) for k,v in batch.items()}

outputs = model(**batch)

print(outputs.logits.shape)
print(outputs.loss)
print(outputs.logits)

## Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

## Train

In [ ]:
trainer.train()

## Save the artifacts

In [ ]:
trainer.save_model(
    "/kaggle/working/model2_deberta_v1"
)

tokenizer.save_pretrained(
    "/kaggle/working/model2_deberta_v1"
)

## Predict Test Logic

In [ ]:
preds = trainer.predict(test_ds)

logits = preds.predictions

In [ ]:
np.save(
    "/kaggle/working/deberta_logits_v1.npy",
    logits
)

## Convert Logits to Top-3

In [ ]:
labels = np.array(
    ["A","B","C","D","E"]
)

top3_predictions = []

for row in logits:

    idx = np.argsort(row)[::-1][:3]

    top3_predictions.append(
        " ".join(labels[idx])
    )

## Submission for Model 2

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": top3_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

# Model 3

In [10]:
train_data = train_df.copy()
test_data = test_df.copy()

print(train_data.shape)
print(test_data.shape)

(2000, 8)
(500, 7)


## Create Knowledge Base

In [11]:
knowledge = []

for _,row in train_data.iterrows():

    answer_text = row[row["answer"]]

    text = (
        row["prompt"]
        +
        " "
        +
        answer_text
    )

    knowledge.append(text)


len(knowledge)

2000

## Load Retriever Model

In [12]:
retriever = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

retriever = retriever.to("cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Encode Knowledge

In [13]:
knowledge_embeddings = retriever.encode(
    knowledge,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(knowledge_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(2000, 384)


## Build FAISS Vector DB

In [14]:
dimension = knowledge_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(knowledge_embeddings)

## Retrieve Context

In [15]:
def retrieve_context(
    question,
    top_k=5
):

    q_emb = retriever.encode(
        [question],
        normalize_embeddings=True
    )

    scores, ids = index.search(
        q_emb,
        top_k
    )

    contexts = [
        knowledge[i]
        for i in ids[0]
    ]

    return " ".join(contexts)

In [16]:
# Test Case
retrieve_context(
    test_data.prompt.iloc[0]
)

'Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. 

## Load Cross Encoder

In [17]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Score MCQ Options

In [18]:
options = [
    "A",
    "B",
    "C",
    "D",
    "E"
]


def predict_question(row):

    context = retrieve_context(
        row["prompt"]
    )

    pairs=[]

    for op in options:

        text = (
            "Context: "
            + context
            +
            " Question: "
            + row["prompt"]
            +
            " Answer: "
            + row[op]
        )

        pairs.append(
            (
                row["prompt"],
                text
            )
        )


    scores = reranker.predict(
        pairs
    )


    order = np.argsort(scores)[::-1]

    prediction = " ".join(
        np.array(options)[order[:3]]
    )

    return prediction

In [23]:
for i in range(5):

    print("QUESTION")
    print(test_data.prompt.iloc[i])

    print("\nCONTEXT")
    print(retrieve_context(test_data.prompt.iloc[i]))

    print("="*80)

QUESTION
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.

CONTEXT
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy. What is the relationship between the Hamiltonians and eigen

## Generate Submission

In [19]:
predictions=[]

for _,row in tqdm(
    test_data.iterrows(),
    total=len(test_data)
):

    predictions.append(
        predict_question(row)
    )

  0%|          | 0/500 [00:00<?, ?it/s]

## Submission for Model 3

In [21]:
submission = pd.DataFrame(
    {
        "id":test_data.id,
        "prediction":predictions
    }
)


submission.to_csv(
    "submission_rag.csv",
    index=False
)

# Model 3 -> Approach 2

In [8]:
train_data = train_df.copy()
test_data = test_df.copy()

## Create Long Format 

In [9]:
options = ["A","B","C","D","E"]

train_rows=[]

for _,row in train_data.iterrows():

    for op in options:

        train_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[op],
            "option_label": op,
            "label": int(op == row["answer"])
        })


train_long = pd.DataFrame(train_rows)

In [10]:
test_rows=[]

for _,row in test_data.iterrows():

    for op in options:

        test_rows.append({
            "id":row["id"],
            "prompt":row["prompt"],
            "option":row[op],
            "option_label":op
        })


test_long=pd.DataFrame(test_rows)

## Clean Prompt for Retrieval

In [12]:
def clean_prompt(text):

    remove = [
        "Pick the best possible answer:",
        "Determine the correct option:",
        "Choose the correct answer:",
        "Select the most accurate option:",
        "Identify the correct statement:"
    ]

    for r in remove:
        text=text.replace(r,"")

    return text.strip()

## Build Knowledge Base

In [13]:
knowledge=[]


for _,row in train_data.iterrows():

    answer_text = row[row["answer"]]

    doc = (
        clean_prompt(row["prompt"])
        +
        " "
        +
        answer_text
    )

    knowledge.append(doc)


knowledge = list(dict.fromkeys(knowledge))

len(knowledge)

1232

## Create FAISS Vector DB

### Retriever

In [14]:
retriever = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Encode KB:

In [15]:
kb_embeddings = retriever.encode(
    knowledge,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

### Build FAISS

In [16]:
dim = kb_embeddings.shape[1]

index = faiss.IndexFlatIP(dim)

index.add(kb_embeddings)

## Retrieve Top 20

In [17]:
def retrieve_docs(question, k=20):

    emb = retriever.encode(
        [clean_prompt(question)],
        normalize_embeddings=True
    )

    scores, ids = index.search(
        emb,
        k
    )

    return [
        knowledge[i]
        for i in ids[0]
    ]

## Cross Encoder Reranking

In [18]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [19]:
def get_best_context(question):

    docs = retrieve_docs(question)

    pairs = [
        [
            clean_prompt(question),
            doc
        ]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    best = np.argmax(scores)

    return docs[best]

## Generate Contexts Once

In [20]:
train_context={}

for qid,prompt in tqdm(
    zip(train_data.id, train_data.prompt),
    total=len(train_data)
):

    train_context[qid] = get_best_context(prompt)

  0%|          | 0/2000 [00:00<?, ?it/s]

In [21]:
test_context={}

for qid,prompt in tqdm(
    zip(test_data.id, test_data.prompt),
    total=len(test_data)
):

    test_context[qid] = get_best_context(prompt)

  0%|          | 0/500 [00:00<?, ?it/s]

In [22]:
train_long["context"] = (
    train_long["id"].map(train_context)
)

test_long["context"] = (
    test_long["id"].map(test_context)
)

## Create RAG Text Features

In [23]:
train_text = (
    train_long.context
    +
    " [SEP] "
    +
    train_long.prompt
    +
    " [SEP] "
    +
    train_long.option
)


test_text = (
    test_long.context
    +
    " [SEP] "
    +
    test_long.prompt
    +
    " [SEP] "
    +
    test_long.option
)

## TF-IDF + Char Features

In [28]:
word_vec = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2)
)

char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=15000
)

X_word_train = word_vec.fit_transform(train_text)
X_word_test = word_vec.transform(test_text)


X_char_train = char_vec.fit_transform(train_text)
X_char_test = char_vec.transform(test_text)

## Add RAG Numerical Features

In [29]:
def overlap(a,b):

    a=set(a.lower().split())
    b=set(b.lower().split())

    return len(a & b)

In [30]:
for df in [train_long,test_long]:

    df["context_overlap"] = df.apply(
        lambda x:
        overlap(
            x.context,
            x.option
        ),
        axis=1
    )


    df["option_len"] = (
        df.option.str.len()
    )

In [31]:
num_cols=[
    "context_overlap",
    "option_len"
]

X_num_train=csr_matrix(
    train_long[num_cols]
)

X_num_test=csr_matrix(
    test_long[num_cols]
)

## Final Matrix

In [32]:
X_train=hstack([
    X_word_train,
    X_char_train,
    X_num_train
]).tocsr()


X_test=hstack([
    X_word_test,
    X_char_test,
    X_num_test
]).tocsr()

## Train LGBMRanker

In [38]:
def map3_score(df):

    scores = []

    for _, g in df.groupby("id"):

        pred = (
            g.sort_values(
                "oof_pred",
                ascending=False
            )["option_label"]
            .tolist()[:3]
        )

        true = g.loc[
            g["label"] == 1,
            "option_label"
        ].iloc[0]

        if true == pred[0]:
            scores.append(1.0)
        elif true == pred[1]:
            scores.append(0.5)
        elif true == pred[2]:
            scores.append(1/3)
        else:
            scores.append(0.0)

    return np.mean(scores)

In [39]:
params = {

    "objective":"lambdarank",
    "metric":"map",
    "learning_rate":0.03,

    "n_estimators":500,

    "num_leaves":31,

    "random_state":42,
    "verbose":-1
}

groups = train_long.id
y = train_long.label

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

gkf = GroupKFold(n_splits=5)

fold_scores = []

for fold, (tr, val) in enumerate(
    gkf.split(X_train, y, groups)
):

    print(f"\n========== Fold {fold+1} ==========")

    model = LGBMRanker(**params)

    train_group = (
        train_long.iloc[tr]
        .groupby("id")
        .size()
        .values
    )

    model.fit(
        X_train[tr],
        y.iloc[tr],
        group=train_group
    )

    # Validation predictions
    oof[val] = model.predict(X_train[val])

    # Test predictions
    test_preds += model.predict(X_test) / gkf.n_splits

    # Compute MAP@3 for this fold
    fold_df = train_long.iloc[val].copy()
    fold_df["oof_pred"] = oof[val]

    score = map3_score(fold_df)
    fold_scores.append(score)

    print(f"Fold {fold+1} MAP@3 : {score:.5f}")


========== Fold 1 ==========
Fold 1 MAP@3 : 0.99625

========== Fold 2 ==========
Fold 2 MAP@3 : 0.99875

========== Fold 3 ==========
Fold 3 MAP@3 : 0.99625

========== Fold 4 ==========
Fold 4 MAP@3 : 0.99750

========== Fold 5 ==========
Fold 5 MAP@3 : 0.99875


## Submission dataset creation

In [35]:
test_long["score"]=test_preds

submission = (
    test_long
    .sort_values(
        ["id","score"],
        ascending=[True,False]
    )
    .groupby("id")
    .option_label
    .apply(
        lambda x:
        " ".join(
            x.head(3)
        )
    )
    .reset_index()
)

## Submission for Model 3 improved

In [36]:
submission.columns=[
    "id",
    "prediction"
]

submission.to_csv(
    "submission.csv",
    index=False
)

# Appendix

## Milestone 1

In [32]:
import numpy as np
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [34]:
print(f"Columns of the train dataset : {train_data.columns}")

Columns of the train dataset : Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [35]:
answer_counts = train_data['answer'].value_counts()
print("Frequency Distribution of the correct answers : \n", answer_counts)

result = answer_counts.max() + answer_counts.min()
print("The sum of the occurrences of the most frequent \
option and the least frequent option:", result)

Frequency Distribution of the correct answers : 
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
The sum of the occurrences of the most frequent option and the least frequent option: 814


2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [36]:
translator = str.maketrans('', '', string.punctuation)

clean_prompts = (
    train_data['prompt']
    .fillna('')
    .str.lower()
    .str.translate(translator)
)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

print(f"Total number of unique words (vocabulary size) across \
the entire cleaned prompt column : {len(vocab)}")

Total number of unique words (vocabulary size) across the entire cleaned prompt column : 859


3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [37]:
train_data.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [38]:
display(train_data[train_data['id']==1])
stop_words  = set(ENGLISH_STOP_WORDS)
words = clean_prompts[0].split()

filtered_words = [word for word in words if word not in stop_words]
print(f"Before filtering words : {words}")
print(f"Filtered Words : {filtered_words}")
print(f"Words are left in the prompt for Row ID 1 after filtering : {len(filtered_words)}")

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B


Before filtering words : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
Filtered Words : ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Words are left in the prompt for Row ID 1 after filtering : 13


4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [39]:
combined_text = (
    train_data['prompt'].fillna('') + ' ' +
    train_data['A'].fillna('') + ' ' +
    train_data['B'].fillna('') + ' ' +
    train_data['C'].fillna('') + ' ' +
    train_data['D'].fillna('') + ' ' +
    train_data['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

vocab_size = len(vectorizer.vocabulary_)

print("Vocabulary size:", vocab_size)

Vocabulary size: 2762


5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [40]:
# Row with id = 1
row = train_data[train_data['id'] == 1].iloc[0]

prompt = row['prompt']
option_a = row['A']

# Used the vectorizer already fitted in Q4
prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(round(similarity, 4))

0.272


6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [41]:
predictions = []

for _, row in train_data.iterrows():

    texts = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]
    vectors = vectorizer.transform(texts)
    prompt_vec = vectors[0]
    option_vecs = vectors[1:]

    sims = cosine_similarity(prompt_vec, option_vecs)[0]

    predicted = ['A', 'B', 'C', 'D', 'E'][np.argmax(sims)]
    predictions.append(predicted)
   

accuracy = (train_data['answer'] == predictions).mean() * 100

print(f"Accuracy = {accuracy:.2f}%")

Accuracy = 13.55%


7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [42]:
def find_MAP(truth,pred):
    score = 0
    if truth == pred[0]:
        score = 1.0
    elif truth == pred[1]:
        score = 1/2
    elif truth == pred[2]:
        score = 1/3
    return score

In [43]:
truth = 'C'
pred = ['C','A','B']
print(f"MAP@3 is : {find_MAP(truth,pred)}")

MAP@3 is : 1.0


8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E? 

In [44]:
def find_MAP(truth,pred):
    score = 0
    if truth == pred[0]:
        score = 1.0
    elif truth == pred[1]:
        score = 1/2
    elif truth == pred[2]:
        score = 1/3
    return score

In [45]:
truth = 'B'
pred = ['D','B','E']
print(f"MAP@3 is : {find_MAP(truth,pred)}")

MAP@3 is : 0.5


9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [46]:
answer_counts = train_data['answer'].value_counts()

print("Answer Frequency Distribution : ")
print(answer_counts)

top3 = answer_counts.index[:3].tolist()

print("Top 3 Answers : ")
print(top3)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0

map3 = train_data['answer'].apply(
    lambda x: apk(x, top3)
).mean()

print(f"Majority Class Baseline MAP@3 = {map3:.3f}")

Answer Frequency Distribution : 
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3 Answers : 
['B', 'C', 'A']
Majority Class Baseline MAP@3 = 0.421


10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [47]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0

scores = []

for _, row in train_data.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = {}

    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    scores.append(
        apk(row['answer'], top3)
    )

map3 = np.mean(scores)

print(f"MAP@3 = {map3:.3f}")

MAP@3 = 0.296


## Milestone 2 

In [51]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [16]:
# Load CSV (no pandas)
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
print(dataset)
# Create combined_text = prompt + " " + A
dataset = dataset.map(
    lambda x: {"combined_text": x["prompt"] + " " + x["A"]}
)

# Character length of row at index 51
length = len(dataset[51]["combined_text"])

print(f"Exact character length of the combined_text string \
for the row at index 51 : {length}")

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})
Exact character length of the combined_text string for the row at index 51 : 614


2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer? 

In [28]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Exact total vocabulary size (the maximum number of unique subword \
tokens the model knows) hardcoded into this tokenizer : {tokenizer.vocab_size}")

Exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer : 30522


3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [18]:
print(f"Exact integer ID assigned to the \
[SEP] : {tokenizer.sep_token_id}")

Exact integer ID assigned to the [SEP] : 102


4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [21]:
encoded = tokenizer(
                   list(dataset["prompt"]),
                   padding='max_length',
                   truncation=True, 
                   max_length=128,
                   return_tensors='pt')
print(f"Exact geometric shape (dimensions) of the resulting \
input_ids tensor : {encoded["input_ids"].shape}")

Exact geometric shape (dimensions) of the resulting input_ids tensor : torch.Size([2000, 128])


5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

    In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [26]:
hidden_embedding_size = 768
attention_heads = 12
# Attention head is divided equally among the attention heads
individual_attention_head_size = hidden_embedding_size//attention_heads
print(f"Eexact dimensionality (size) of each individual \
attention head : {individual_attention_head_size}")

Eexact dimensionality (size) of each individual attention head : 64


6.Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [30]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Row index 0 prompt
text = dataset[0]["prompt"]

# Default tokenization (no padding/truncation)
inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Forward pass
outputs = model(**inputs)

print(f"Exact shape of the last_hidden_state tensor \
returned : {outputs.last_hidden_state.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Exact shape of the last_hidden_state tensor returned : torch.Size([1, 31, 768])


7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [32]:
cls_embedding = outputs.last_hidden_state[0, 0, :]

# first 5 values
first_5 = cls_embedding[:5]

print(first_5)

# sum and round
answer = round(first_5.sum().item(), 4)

print(f"Sum of the first 5 float values in \
this [CLS] vector : {answer}")

tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480], grad_fn=<SliceBackward0>)
Sum of the first 5 float values in this [CLS] vector : -1.2001


8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

   What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [33]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Check tokens
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

# Find fusion index
fusion_idx = tokens.index("fusion")

print("fusion index:", fusion_idx)

with torch.no_grad():
    outputs = model(**inputs)

# last layer attention
last_attention = outputs.attentions[-1]

print(last_attention.shape)

# batch 0, head 0, CLS token attention to fusion
attention_value = last_attention[0, 0, 0, fusion_idx]

answer = round(attention_value.item(), 4)

print(f"Exact attention weight (a float value) \
that the [CLS] token (token index 0) pays to the word fusion : {answer}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion index: 4
torch.Size([1, 12, 10, 10])
Exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion : 0.1025


9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [53]:
# Load dataset (no pandas)
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

# Initialize model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row ID 0
prompt_text = dataset[0]["prompt"]
option_b_text = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt_text)
option_b_embedding = model.encode(option_b_text)

# Cosine similarity using sentence_transformers.util.cos_sim()
similarity = util.cos_sim(
    prompt_embedding,
    option_b_embedding
)

answer = round(similarity.item(), 4)

print(f"The resulting similarity score : {answer}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The resulting similarity score : 0.7658


10. Build two complete ranking pipelines evaluating every row in train.csv.

    Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.
    
    Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.
    
    First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 
    
    Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [56]:
# -----------------------------
# Load data
# -----------------------------

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]


options = ["A", "B", "C", "D", "E"]


# -----------------------------
# MAP@3 function
# -----------------------------

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)

    return 0


def map3(actuals, predictions):
    scores = []

    for actual, pred in zip(actuals, predictions):
        scores.append(apk(actual, pred))

    return np.mean(scores)



# =================================================
# Pipeline 1: TF-IDF cosine similarity
# =================================================

all_text = []

for row in dataset:
    all_text.append(row["prompt"])
    
    for op in options:
        all_text.append(row[op])


tfidf = TfidfVectorizer()

tfidf.fit(all_text)


tfidf_predictions = []


for row in dataset:

    prompt_vec = tfidf.transform(
        [row["prompt"]]
    )

    sims = {}

    for op in options:

        option_vec = tfidf.transform(
            [row[op]]
        )

        sims[op] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]


    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    tfidf_predictions.append(
        ranked[:3]
    )


# =================================================
# Pipeline 2: MiniLM embeddings
# =================================================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


minilm_predictions = []


for row in dataset:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_texts = [
        row[o] for o in options
    ]

    option_embeddings = model.encode(
        option_texts,
        convert_to_tensor=True
    )


    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]


    top_indices = similarities.argsort(
        descending=True
    )


    ranked_options = [
        options[i]
        for i in top_indices
    ]


    minilm_predictions.append(
        ranked_options[:3]
    )



# =================================================
# Results
# =================================================

answers = dataset["answer"]


minilm_map3 = map3(
    answers,
    minilm_predictions
)


print(
    "MiniLM MAP@3:",
    round(minilm_map3, 3)
)



# Count questions fixed by MiniLM

fixed_count = 0


for actual, tfidf_pred, mini_pred in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if (
        actual not in tfidf_pred
        and actual in mini_pred
    ):
        fixed_count += 1


print(
    "TF-IDF missed but MiniLM got:",
    fixed_count
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3: 0.423
TF-IDF missed but MiniLM got: 531


11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [55]:
# Load train.csv
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]


# Initialize zero-shot pipeline
classifier = pipeline(
    "zero-shot-classification"
)


# Row index 1 (2nd row)
prompt_text = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]


# Prediction
result = classifier(
    prompt_text,
    candidate_labels
)


print(result)


# Top ranked probability score
top_score = result["scores"][0]

print(f"The probability score given to the top-ranked\
option : {round(top_score, 4)}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

    What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [45]:
classifier = pipeline(
    "zero-shot-classification"
)


prompt_text = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]


# -----------------------------
# Softmax (previous question)
# -----------------------------

softmax_result = classifier(
    prompt_text,
    candidate_labels
)

softmax_sum = sum(
    softmax_result["scores"]
)


# -----------------------------
# Independent Sigmoid
# -----------------------------

sigmoid_result = classifier(
    prompt_text,
    candidate_labels,
    multi_label=True
)

sigmoid_sum = sum(
    sigmoid_result["scores"]
)


# Difference

difference = abs(
    softmax_sum - sigmoid_sum
)

print("Softmax scores:", softmax_result["scores"])
print("Softmax sum:", softmax_sum)

print("Sigmoid scores:", sigmoid_result["scores"])
print("Sigmoid sum:", sigmoid_sum)

print("Difference:", round(difference, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax scores: [0.4574522376060486, 0.2750644385814667, 0.26748329401016235]
Softmax sum: 0.9999999701976776
Sigmoid scores: [0.00046926175127737224, 2.063553074549418e-05, 1.9700279153767042e-05]
Sigmoid sum: 0.0005095975611766335
Difference: 0.9995


13. Let's try Generative AI instead of Classification. 

    Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
    Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [52]:
# Load FLAN-T5
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

# Row index 0
row = dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Generate
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5
    )

# Decode
generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("The exact string output returned by the model:", generated_text)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The exact string output returned by the model: B
